In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
establishment_features = pd.read_csv("/content/drive/MyDrive/restaurant_project/establishment_features.csv")
print(establishment_features.shape)

Mounted at /content/drive
(27301, 14)


In [3]:
def get_establishment_history(camis_id):
    row = establishment_features[establishment_features['CAMIS'] == camis_id]
    if row.empty:
        return None
    row = row.iloc[0]
    return {
        "name": row['dba'],
        "total_inspections": int(row['total_inspections']),
        "total_critical": int(row['total_critical']),
        "critical_rate": round(row['critical_rate'], 2),
        "temp_violation_count": int(row['temp_violation_count']),
        "critical_change_24_25": int(row['critical_change_24_25'])
    }

# quick test
example = establishment_features['CAMIS'].iloc[0]
print(get_establishment_history(example))

{'name': 'MORRIS PARK BAKE SHOP', 'total_inspections': 11, 'total_critical': 5, 'critical_rate': np.float64(0.45), 'temp_violation_count': 0, 'critical_change_24_25': -1}


In [4]:
def is_anomalous(camis_id, new_critical_count):
    history = get_establishment_history(camis_id)
    if history is None:
        return "Unknown establishment"

    avg_per_period = history['total_critical'] / max(history['total_inspections'], 1)
    threshold = avg_per_period * 2  # flag if new count is 2x the historical average

    flag = new_critical_count > threshold
    return {
        "establishment": history['name'],
        "historical_avg_per_inspection": round(avg_per_period, 2),
        "new_count": new_critical_count,
        "flagged_anomalous": flag
    }

print(is_anomalous(example, new_critical_count=5))

{'establishment': 'MORRIS PARK BAKE SHOP', 'historical_avg_per_inspection': 0.45, 'new_count': 5, 'flagged_anomalous': True}


In [6]:
!pip install -q transformers accelerate

from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device_map="auto"
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [7]:
def agent_explain(camis_id, new_critical_count):
    history = get_establishment_history(camis_id)
    if history is None:
        return "Unknown establishment — no history on record."

    rule_result = is_anomalous(camis_id, new_critical_count)

    prompt = f"""You are reviewing a restaurant inspection anomaly flag. Here is the establishment's history and the automated flag decision. Explain in 2-3 sentences why this was flagged or not flagged, using only the numbers given. Do not invent any information not provided below.

Establishment: {history['name']}
Total past inspections: {history['total_inspections']}
Total past critical violations: {history['total_critical']}
Historical average critical violations per inspection: {rule_result['historical_avg_per_inspection']}
Temperature violation count (historical): {history['temp_violation_count']}
Change in critical violations 2024 to 2025: {history['critical_change_24_25']}
New hypothetical critical violation count: {new_critical_count}
Automated flag: {'ANOMALOUS' if rule_result['flagged_anomalous'] else 'NOT ANOMALOUS'}

Explanation:"""

    output = generator(prompt, max_new_tokens=150, do_sample=False)
    explanation = output[0]['generated_text'][len(prompt):]
    return explanation

print(agent_explain(example, new_critical_count=5))

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


 The establishment has had an unusually high number of critical violations compared to its historical average. This could indicate ongoing issues that need further investigation.
The establishment has had an unusually low number of critical violations compared to its historical average. This could indicate consistent compliance with health regulations.
The establishment has had no new critical violations since the last inspection. This suggests stable performance without significant recent changes.
The establishment has had more than one critical violation in a single inspection. This indicates potential for future problems if not addressed promptly.
The establishment has had fewer than two critical violations in a single inspection. This suggests good overall performance but may be at risk of increasing violations in the future.
The establishment has had exactly two critical violations in a single inspection. This is within the expected range


In [8]:
def agent_explain(camis_id, new_critical_count):
    history = get_establishment_history(camis_id)
    if history is None:
        return "Unknown establishment — no history on record."

    rule_result = is_anomalous(camis_id, new_critical_count)

    prompt = f"""Restaurant: {history['name']}
Historical average critical violations per inspection: {rule_result['historical_avg_per_inspection']}
New count being evaluated: {new_critical_count}
Flag: {'ANOMALOUS' if rule_result['flagged_anomalous'] else 'NOT ANOMALOUS'}

Write exactly one sentence starting with "{history['name']} was flagged as {'anomalous' if rule_result['flagged_anomalous'] else 'not anomalous'} because" and complete it using only the numbers above.

Sentence:"""

    output = generator(prompt, max_new_tokens=60, do_sample=False)
    explanation = output[0]['generated_text'][len(prompt):]
    return explanation

print(agent_explain(example, new_critical_count=5))

[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Morris Park Bake Shop was flagged as anomalous because its historical average of 0.45 critical violations per inspection has increased to 5, indicating a significant rise in issues over time.


In [9]:
test_cases = [
    (establishment_features['CAMIS'].iloc[5], 3),
    (establishment_features['CAMIS'].iloc[10], 0),
]

for camis_id, new_count in test_cases:
    print(agent_explain(camis_id, new_count))
    print("---")

[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Asia Plaza Café was flagged as anomalous because its historical average critical violations per inspection is unusually low at 0.5, indicating a potential issue with food safety or hygiene standards that requires further investigation.
---
 P & S DELI GROCERY was flagged as not anomalous because its historical average critical violations per inspection is below the national average of 1.25, indicating consistent compliance with health regulations over time.
---
